**Week-5 Assignment**



**setup:**

In [31]:
!apt-get install -y openjdk-11-jdk -q
!pip install pyspark==3.5.1 -q

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, TimestampType

spark = SparkSession.builder.appName("Week5").master("local[*]").getOrCreate()
print("Spark ready:", spark.version)

Reading package lists...
Building dependency tree...
Reading state information...
openjdk-11-jdk is already the newest version (11.0.31+11-1ubuntu1~22.04.2).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
Spark ready: 3.5.1


In [32]:
from google.colab import files
uploaded = files.upload()

Saving week5_dataset_1000.csv to week5_dataset_1000 (3).csv


In [33]:
schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("email", StringType(), True),
    StructField("username", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("subscription", StringType(), True),
    StructField("transaction_date", StringType(), True),
    StructField("city", StringType(), True),
    StructField("region", StringType(), True),
    StructField("product_category", StringType(), True),
    StructField("price", FloatType(), True),
    StructField("status", StringType(), True),
    StructField("store_id", StringType(), True),
    StructField("raw_timestamp", StringType(), True),
])

df = spark.read.csv("week5_dataset_1000.csv", header=True, schema=schema)
print(f"Loaded {df.count()} rows")
df.show(5, truncate=False)

Loaded 1000 rows
+-------+-------------------+--------+---+------------+----------------+-----------+------+----------------+-------+--------+--------+-------------------+
|user_id|email              |username|age|subscription|transaction_date|city       |region|product_category|price  |status  |store_id|raw_timestamp      |
+-------+-------------------+--------+---+------------+----------------+-----------+------+----------------+-------+--------+--------+-------------------+
|655    |user655@example.com|user_655|30 |Premium     |2023-05-23      |Chicago    |West  |Toys            |NULL   |active  |S007    |2023-08-27 16:38:00|
|28     |user28@example.com |user_28 |59 |Standard    |2024-03-05      |Phoenix    |South |Sports          |19.68  |inactive|S014    |2023-12-15 08:09:00|
|221    |user221@example.com|user_221|20 |Basic       |2023-04-10      |Dallas     |North |Sports          |1097.3 |active  |S013    |2023-03-22 17:18:00|
|850    |user850@example.com|user_850|38 |Standard   

Q1) What are the key limitations of traditional MapReduce that make Spark a preferred choice for modern big data processing?


ans:-
MapReduce has 4 key limitations that Spark solves:

1. Disk I/O Bottleneck
   MapReduce writes intermediate results to HDFS after every step.
   Spark keeps data in memory between steps so no disk read/write happens.

2. No In-Memory Caching
   MapReduce re-reads the full dataset from disk on every iteration.
   Spark uses df.cache() to store data in RAM and only reads from disk once.

3. High Latency
   MapReduce starts a new JVM and schedules tasks from scratch for every job.
   Spark builds a DAG of all steps and executes them together in one optimised run.

4. Rigid Two-Phase Model
   MapReduce only supports Map then Reduce.
   Spark supports SQL, ML, graph and streaming all in one unified engine.

demo:-cache() vs disk read

In [34]:
df_cached = df.select("user_id", "price").cache()
_ = df_cached.count()

import time
print("Simulating MapReduce (no cache) vs Spark (cached):\n")

for i in range(1, 4):
    t0 = time.time()
    result = df_cached.agg(F.avg("price")).collect()[0][0]
    ms =(time.time() - t0) * 1000
    print(f"  Iteration {i}: avg(price) = {result:.2f}  [{ms:.1f} ms — from RAM]")

df_cached.unpersist()

print("\nMapReduce would read HDFS 3 times. Spark reads disk once.")
print("Done")

Simulating MapReduce (no cache) vs Spark (cached):

  Iteration 1: avg(price) = 757.66  [294.7 ms — from RAM]
  Iteration 2: avg(price) = 757.66  [399.0 ms — from RAM]
  Iteration 3: avg(price) = 757.66  [331.5 ms — from RAM]

MapReduce would read HDFS 3 times. Spark reads disk once.
Done


In [35]:
print(spark.version)
print(df.count())

3.5.1
1000


Q2)Explain how Spark uses In-Memory Computing to speed up iterative machine learning algorithms compared to disk-based systems.


In [36]:
import time

df_features = df.select("user_id", "age", "price").cache()
df_features.count()

for i in range(1, 6):
    t0 = time.time()
    avg = df_features.agg(F.avg("price")).collect()[0][0]
    ms = (time.time() - t0) * 1000
    note = "disk read" if i == 1 else "from RAM"
    print(f"Iteration {i}: avg price = {avg:.2f}, time = {ms:.1f} ms ({note})")

df_features.unpersist()

Iteration 1: avg price = 757.66, time = 433.4 ms (disk read)
Iteration 2: avg price = 757.66, time = 226.6 ms (from RAM)
Iteration 3: avg price = 757.66, time = 232.9 ms (from RAM)
Iteration 4: avg price = 757.66, time = 317.1 ms (from RAM)
Iteration 5: avg price = 757.66, time = 246.5 ms (from RAM)


DataFrame[user_id: int, age: int, price: float]

Iterative ML algorithms repeat the same computation on the same dataset many times.

In MapReduce:
   Each iteration reads from HDFS, computes, then writes back to HDFS.
   100 iterations = 200 disk read/write cycles.

In Spark:
   df.cache() loads the dataset into RAM once.
   Every iteration after the first reads from memory, not disk.
   100 iterations = 1 disk read + 99 RAM reads.

RAM is roughly 100x faster than SSD which is why Spark made large scale ML practical.

Q3)Write a code snippet to remove all duplicate rows from a DataFrame based on a specific set of columns: user_id and transaction_date.


In [37]:
before = df.count()
df_deduped = df.dropDuplicates(subset=["user_id", "transaction_date"])
after = df_deduped.count()

print(f"Before: {before} rows")
print(f"After: {after} rows")
print(f"Removed: {before - after} duplicate rows")

df_deduped.select("user_id", "username", "transaction_date", "price").show(10, truncate=False)

Before: 1000 rows
After: 1000 rows
Removed: 0 duplicate rows
+-------+--------+----------------+-------+
|user_id|username|transaction_date|price  |
+-------+--------+----------------+-------+
|1      |user_1  |2023-11-11      |660.46 |
|1      |user_1  |2024-04-28      |1423.01|
|4      |user_4  |2023-08-07      |570.11 |
|4      |user_4  |2024-05-25      |887.4  |
|7      |user_7  |2023-11-27      |880.72 |
|7      |user_7  |2024-07-23      |578.84 |
|8      |user_8  |2024-05-12      |1025.97|
|9      |user_9  |2024-09-29      |745.11 |
|10     |user_10 |2024-05-15      |NULL   |
|11     |user_11 |2024-09-26      |499.01 |
+-------+--------+----------------+-------+
only showing top 10 rows



Q4)Given a DataFrame df_sales, write a query to filter for rows where the region is 'West' and then group by product_category to find the average sale_amount.


In [38]:
result = (df
    .filter(F.col("region") == "West")
    .groupBy("product_category")
    .agg(F.avg("price").alias("avg_sale_amount"), F.count("*").alias("record_count"))
    .orderBy(F.col("avg_sale_amount").desc()))

result.show(truncate=False)

+----------------+-----------------+------------+
|product_category|avg_sale_amount  |record_count|
+----------------+-----------------+------------+
|Books           |815.8896549487936|32          |
|Electronics     |732.2176244826544|21          |
|Clothing        |693.0592486381531|41          |
|Toys            |685.6962162224022|39          |
|Beauty          |675.1825967011629|29          |
|Furniture       |670.3796589292328|30          |
|Sports          |602.05928339277  |31          |
|Food            |593.8827560030181|31          |
+----------------+-----------------+------------+



Q5)What is the difference between .na.drop() and .na.fill()? Provide a code example of filling null values in a status column with the string 'Unknown'.


In [39]:
null_count = df.filter(F.col("status").isNull()).count()
print(f"Null status values: {null_count}")

df_dropped = df.na.drop()
print(f"na.drop() -> {df_dropped.count()} rows")

df_filled = df.na.fill({"status": "Unknown"})
print(f"na.fill() -> {df_filled.count()} rows")

df_filled.groupBy("status").count().orderBy(F.col("count").desc()).show()

Null status values: 255
na.drop() -> 652 rows
na.fill() -> 1000 rows
+--------+-----+
|  status|count|
+--------+-----+
| Unknown|  255|
| pending|  254|
|inactive|  253|
|  active|  238|
+--------+-----+



na.drop()
   Removes entire rows that contain null values.
   Use when a null makes the row useless, like a null user ID or null key column.
   df.na.drop(subset=["email"]) only removes rows where email is null.

na.fill()
   Replaces nulls with a default value. All rows are kept.
   Use when null means not provided and a sensible default exists.
   df.na.fill({"status": "Unknown", "price": 0})

Key point:
   na.drop() reduces your row count. na.fill() keeps it the same.
   The choice is a business decision as both give different aggregation results.

Q6) Write a query to find the total count of records for each city in a DataFrame, but only for cities where the count is greater than 100.


In [40]:
result = (df
    .groupBy("city")
    .count()
    .filter(F.col("count") > 100)
    .orderBy(F.col("count").desc()))

result.show(truncate=False)

+-------+-----+
|city   |count|
+-------+-----+
|Chicago|123  |
|Denver |111  |
|Miami  |102  |
|Dallas |101  |
|Boston |101  |
+-------+-----+



Q7)How does the immutability of Spark DataFrames affect how you perform "data cleaning" steps like dropping columns or renaming them?


In [41]:
print("Original columns:", df.columns)

df_clean = (df
    .drop("raw_timestamp")
    .withColumnRenamed("price", "sale_amount")
    .withColumnRenamed("product_category", "category"))

print("Cleaned columns:", df_clean.columns)
print("Original unchanged:", "raw_timestamp" in df.columns)

df_clean.show(5, truncate=False)

Original columns: ['user_id', 'email', 'username', 'age', 'subscription', 'transaction_date', 'city', 'region', 'product_category', 'price', 'status', 'store_id', 'raw_timestamp']
Cleaned columns: ['user_id', 'email', 'username', 'age', 'subscription', 'transaction_date', 'city', 'region', 'category', 'sale_amount', 'status', 'store_id']
Original unchanged: True
+-------+-------------------+--------+---+------------+----------------+-----------+------+--------+-----------+--------+--------+
|user_id|email              |username|age|subscription|transaction_date|city       |region|category|sale_amount|status  |store_id|
+-------+-------------------+--------+---+------------+----------------+-----------+------+--------+-----------+--------+--------+
|655    |user655@example.com|user_655|30 |Premium     |2023-05-23      |Chicago    |West  |Toys    |NULL       |active  |S007    |
|28     |user28@example.com |user_28 |59 |Standard    |2024-03-05      |Phoenix    |South |Sports  |19.68      

Spark DataFrames are immutable. Every transformation returns a new DataFrame.
The original is never modified.

What this means for data cleaning:
   You must always assign the result back: df = df.drop("raw_col")
   Just calling df.drop("raw_col") without assigning does nothing.

Benefits:
   Spark builds a logical plan (DAG) of all chained steps and the Catalyst optimizer
   combines and reorders them before execution, making chains of transformations efficient.
   The original df is always safe so you can create multiple versions from the same source.

Q8)Write a Spark command to filter a dataset for rows where the age is between 18 and 30 (inclusive) and the subscription is 'Premium'.


In [42]:
df_filtered = df.filter((F.col("age").between(18, 30)) & (F.col("subscription") == "Premium"))

print(f"Total rows: {df.count()}")
print(f"Filtered rows: {df_filtered.count()}")

df_filtered.select("user_id", "username", "age", "subscription", "city").show(10, truncate=False)

Total rows: 1000
Filtered rows: 87
+-------+--------+---+------------+-----------+
|user_id|username|age|subscription|city       |
+-------+--------+---+------------+-----------+
|655    |user_655|30 |Premium     |Chicago    |
|498    |user_498|18 |Premium     |New York   |
|470    |user_470|21 |Premium     |Phoenix    |
|528    |user_528|20 |Premium     |Los Angeles|
|713    |user_713|30 |Premium     |Houston    |
|283    |user_283|27 |Premium     |Seattle    |
|664    |user_664|24 |Premium     |Houston    |
|422    |user_422|25 |Premium     |Denver     |
|661    |user_661|27 |Premium     |Chicago    |
|796    |user_796|18 |Premium     |Denver     |
+-------+--------+---+------------+-----------+
only showing top 10 rows



Q9)When cleaning a dataset, why is it often better to handle null values before performing mathematical aggregations like sum() or avg()?

In [43]:
total = df.count()
null_cnt = df.filter(F.col("price").isNull()).count()
print(f"Total rows: {total}, Null prices: {null_cnt} ({null_cnt/total*100:.1f}%)")

avg_default = df.agg(F.avg("price")).collect()[0][0]
avg_filled = df.na.fill({"price": 0}).agg(F.avg("price")).collect()[0][0]
avg_dropped = df.na.drop(subset=["price"]).agg(F.avg("price")).collect()[0][0]

print(f"Default (ignore nulls): {avg_default:.2f}")
print(f"After na.fill(price=0): {avg_filled:.2f}")
print(f"After na.drop():        {avg_dropped:.2f}")

Total rows: 1000, Null prices: 63 (6.3%)
Default (ignore nulls): 757.66
After na.fill(price=0): 709.92
After na.drop():        757.66


Spark's avg() and sum() silently skip null values which creates hidden problems.

avg() divides by the count of non-null rows, not total rows.
   If 10% of prices are null, the average is calculated on only 90% of the data
   with no warning or error.

sum() on an all-null column returns null, not 0.
   This silently breaks any downstream calculation expecting a number.

Best practice:
   Always check null counts first, then decide your strategy before aggregating.
   na.fill({"price": 0}) treats missing as zero.
   na.drop(subset=["price"]) excludes incomplete rows.
   Both are valid but give different results so the choice must be deliberate.

Q10) Write the code to revise a column named raw_timestamp by casting it to a TimestampType and renaming it to event_time.

In [44]:
df_ts = (df
    .withColumn("event_time", F.col("raw_timestamp").cast(TimestampType()))
    .drop("raw_timestamp"))

df_ts.printSchema()
df_ts.select("user_id", "event_time").show(8, truncate=False)

failures = df_ts.filter(F.col("event_time").isNull()).count()
print(f"Parse failures: {failures}")

root
 |-- user_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- price: float (nullable = true)
 |-- status: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- event_time: timestamp (nullable = true)

+-------+-------------------+
|user_id|event_time         |
+-------+-------------------+
|655    |2023-08-27 16:38:00|
|28     |2023-12-15 08:09:00|
|221    |2023-03-22 17:18:00|
|850    |2024-01-25 08:29:00|
|651    |2023-10-04 20:44:00|
|571    |2024-11-02 15:25:00|
|659    |2024-06-05 15:05:00|
|774    |2023-01-12 21:46:00|
+-------+-------------------+
only showing top 8 rows

Parse failures: 0


Q11)Explain the "Shuffle" process that occurs during a grouping operation. Why is it considered a wide transformation?

In [45]:
df.groupBy("city").count().explain()

print(f"Input partitions: {df.rdd.getNumPartitions()}")
df_grouped = df.groupBy("city").count()
print(f"Output partitions: {df_grouped.rdd.getNumPartitions()}")
df_grouped.orderBy(F.col("count").desc()).show(truncate=False)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[city#778], functions=[count(1)])
   +- Exchange hashpartitioning(city#778, 200), ENSURE_REQUIREMENTS, [plan_id=1860]
      +- HashAggregate(keys=[city#778], functions=[partial_count(1)])
         +- FileScan csv [city#778] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/week5_dataset_1000.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<city:string>


Input partitions: 1
Output partitions: 1
+-----------+-----+
|city       |count|
+-----------+-----+
|Chicago    |123  |
|Denver     |111  |
|Miami      |102  |
|Dallas     |101  |
|Boston     |101  |
|Seattle    |99   |
|Houston    |97   |
|New York   |96   |
|Phoenix    |93   |
|Los Angeles|77   |
+-----------+-----+



Narrow Transformation (filter, select, withColumn):
   Each output partition reads from exactly one input partition.
   No data moves between executors. Fast.

Wide Transformation (groupBy, join, distinct, orderBy):
   Each output partition needs data from multiple input partitions.
   Spark must shuffle data across the network to bring matching keys together.

How a Shuffle works:
   1. Each executor writes rows sorted by key to local disk.
   2. Executors fetch the partitions they need from other machines over the network.
   3. Aggregation runs on the fully assembled groups.

Shuffles are the biggest performance bottleneck in Spark.
Reduce them by filtering early, using broadcast joins for small tables
and avoiding unnecessary distinct() and orderBy() calls.

Q12)Write a code snippet that identifies and removes rows where the email column contains null values OR the username is an empty string.

In [46]:
print(f"Null emails: {df.filter(F.col('email').isNull()).count()}")
print(f"Empty usernames: {df.filter(F.col('username') == '').count()}")

bad_rows = (F.col("email").isNull() | (F.col("username") == "") | F.col("username").isNull())
df_clean = df.filter(~bad_rows)

print(f"Before: {df.count()} rows")
print(f"After: {df_clean.count()} rows")

df_clean.select("user_id", "email", "username").show(10, truncate=False)

Null emails: 42
Empty usernames: 0
Before: 1000 rows
After: 919 rows
+-------+-------------------+--------+
|user_id|email              |username|
+-------+-------------------+--------+
|655    |user655@example.com|user_655|
|28     |user28@example.com |user_28 |
|221    |user221@example.com|user_221|
|850    |user850@example.com|user_850|
|651    |user651@example.com|user_651|
|571    |user571@example.com|user_571|
|659    |user659@example.com|user_659|
|118    |user118@example.com|user_118|
|109    |user109@example.com|user_109|
|115    |user115@example.com|user_115|
+-------+-------------------+--------+
only showing top 10 rows



Q13)How do you use the .agg() function to calculate multiple statistics at once, such as the min, max, and mean of the price column?

In [47]:
result = df.groupBy("product_category").agg(
    F.count("price").alias("count"),
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.avg("price").alias("avg_price"),
    F.stddev("price").alias("stddev_price")
).orderBy(F.col("avg_price").desc())

result.show(truncate=False)

overall = df.agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.avg("price").alias("avg_price"),
    F.sum("price").alias("total_revenue")
)
overall.show(truncate=False)

+----------------+-----+---------+---------+-----------------+------------------+
|product_category|count|min_price|max_price|avg_price        |stddev_price      |
+----------------+-----+---------+---------+-----------------+------------------+
|Furniture       |126  |24.49    |1499.73  |790.4707169457088|425.25416372912235|
|Sports          |111  |19.68    |1486.6   |767.2265773635727|417.60939877854184|
|Toys            |144  |12.5     |1494.22  |763.9656953811646|441.1003406784022 |
|Beauty          |115  |13.98    |1491.86  |763.1857392684273|444.8396297745587 |
|Food            |104  |12.35    |1496.75  |759.2306735882393|439.278661259844  |
|Electronics     |105  |39.47    |1492.69  |755.4336208888462|438.68092984603015|
|Books           |105  |13.83    |1477.98  |743.4997161683582|436.3591913155609 |
|Clothing        |127  |22.63    |1494.69  |716.8303136900654|412.64269710604543|
+----------------+-----+---------+---------+-----------------+------------------+

+---------+----

Q14)In the context of cleaning a dataset, what is the risk of using inferSchema=true when your source data contains messy or inconsistent date formats?

In [48]:
messy = spark.createDataFrame([
    ("2024-01-15",), ("01/20/2024",), ("Jan 25 2024",), ("20240210",)
], ["raw_date"])

parsed = (messy
    .withColumn("iso_parse", F.to_timestamp(F.col("raw_date"), "yyyy-MM-dd"))
    .withColumn("us_parse", F.to_timestamp(F.col("raw_date"), "MM/dd/yyyy"))
    .withColumn("mon_parse", F.to_timestamp(F.col("raw_date"), "MMM dd yyyy")))

parsed.show(truncate=False)

df_infer = spark.read.csv("week5_dataset_1000.csv", header=True, inferSchema=True)
print("inferSchema=True:")
df_infer.printSchema()

print("Explicit schema:")
df.printSchema()

+-----------+-------------------+-------------------+-------------------+
|raw_date   |iso_parse          |us_parse           |mon_parse          |
+-----------+-------------------+-------------------+-------------------+
|2024-01-15 |2024-01-15 00:00:00|NULL               |NULL               |
|01/20/2024 |NULL               |2024-01-20 00:00:00|NULL               |
|Jan 25 2024|NULL               |NULL               |2024-01-25 00:00:00|
|20240210   |NULL               |NULL               |NULL               |
+-----------+-------------------+-------------------+-------------------+

inferSchema=True:
root
 |-- user_id: integer (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- price: double 

inferSchema=True tells Spark to scan a sample of rows and guess column types automatically.

Risks with messy date formats:

1. Sampling bias: Spark only scans a fraction of rows.
   Mixed formats in the rest of the data lead to the wrong type being inferred.

2. Silent nulls: If the inferred type does not match some rows,
   those rows become null silently with no error or warning.

3. Falls back to StringType: When formats are too mixed,
   Spark gives up and keeps the column as a string with no parsing at all.

4. Extra scan: inferSchema does a full pass over the data just to detect types,
   which doubles read time on large datasets.

Best practice:
   Define schema explicitly with StructType and keep date columns as StringType.
   Then parse manually with F.to_timestamp(col, "yyyy-MM-dd") so you control failures.

Q15)Write a final processing pipeline that:

1)Filters out duplicates.

2)Fills null prices with 0.

3)Groups by store_id to calculate total revenue.

In [49]:
print(f"Total rows: {df.count()}")
print(f"Duplicates: {df.count() - df.dropDuplicates().count()}")
print(f"Null prices: {df.filter(F.col('price').isNull()).count()}")

result = (df
    .dropDuplicates()
    .na.fill({"price": 0})
    .groupBy("store_id")
    .agg(
        F.sum("price").alias("total_revenue"),
        F.count("*").alias("transaction_count"),
        F.avg("price").alias("avg_transaction_value")
    )
    .orderBy(F.col("total_revenue").desc()))

result.show(truncate=False)

grand = result.agg(F.sum("total_revenue")).collect()[0][0]
print(f"Grand total revenue: ${grand:,.2f}")

spark.stop()

Total rows: 1000
Duplicates: 0
Null prices: 63
+--------+------------------+-----------------+---------------------+
|store_id|total_revenue     |transaction_count|avg_transaction_value|
+--------+------------------+-----------------+---------------------+
|S004    |45334.40018463135 |59               |768.379664146294     |
|S005    |45242.599992752075|60               |754.0433332125345    |
|S015    |42458.57026672363 |53               |801.105099372144     |
|S003    |42316.75          |55               |769.3954545454545    |
|S018    |40912.26996612549 |58               |705.3839649331981    |
|S013    |40755.040100097656|55               |741.0007290926847    |
|S011    |37324.38011550903 |53               |704.2335870850761    |
|S014    |37164.00012397766 |52               |714.6923100764935    |
|S009    |36751.689975738525|51               |720.6213720733044    |
|S010    |36530.77993774414 |48               |761.0579153696696    |
|S020    |34985.18995857239 |51            